In [1]:
import pandas as pd
import json
from pydantic import BaseModel
from openai import OpenAI
import os

In [2]:
# --- Schema Definition ---
class AgentOutput(BaseModel):
    recommended_plan_id: str
    discount_applied_flag: bool
    final_discount_amount: float
    agent_reasoning: str

In [3]:
# --- Tool: Query Customer DB ---
def query_customer_db(user_id: str) -> dict:
    """Simulates the Usage Analyst role querying the database."""

    logs = pd.read_csv('../data/tower_logs.csv')
    subs = pd.read_csv('../data/customer_subscriptions.csv')
    transcripts = pd.read_csv('../data/transcripts.csv')
    
    user_log = logs[logs['user_id'] == user_id].iloc[0]
    user_sub = subs[subs['user_id'] == user_id].iloc[0]
    user_transcripts = transcripts[transcripts['user_id'] == user_id]['text_transcript'].tolist()
    
    return {
        "usage_gb": float(user_log['avg_data_gb']),
        "current_plan": user_sub['current_plan_id'],
        "dropped_calls": int(user_log['dropped_calls']),
        "complaints": user_transcripts
    }

In [4]:
# --- Helper: Load Prompt ---
def load_system_prompt(filepath: str = "prompt.txt") -> str:
    """Loads the external prompt asset."""
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing prompt file at {filepath}")
    with open(filepath, "r") as f:
        return f.read()

In [5]:
# --- Agentic Workflow ---
def run_agentic_workflow(user_id: str) -> str:
    """Executes the prompt-driven pipeline using local Ollama LLM."""
    
    # Gather Data (Usage Analyst)
    customer_data = query_customer_db(user_id)
    
    system_prompt = load_system_prompt("prompt.txt")

    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")    
    response = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Customer Data: {json.dumps(customer_data)}"}
        ],
        response_format={"type": "json_object"},
        temperature=0.1
    )
    
    raw_output = response.choices[0].message.content
    llm_decision = json.loads(raw_output)
    
    # Apply Business Logic Guardrail
    proposed_discount = float(llm_decision.get("proposed_discount_amount", 0.0))
    final_discount = proposed_discount
    
    if proposed_discount > 20.0:
        final_discount = 20.0
        llm_decision["agent_reasoning"] += f" [GUARDRAIL TRIGGERED: AI proposed ${proposed_discount}, reduced to max $20.]"
    
    # Finalize state for Pydantic Validation
    llm_decision["final_discount_amount"] = final_discount
    llm_decision.pop("proposed_discount_amount", None)
    
    # Enforce output schema
    final_output = AgentOutput(**llm_decision)
    return final_output.model_dump_json(indent=2)

In [7]:
print("\nTesting CUST_005:")
print(run_agentic_workflow("CUST_005"))


Testing CUST_005:
{
  "recommended_plan_id": "P_ULTRA",
  "discount_applied_flag": true,
  "final_discount_amount": 20.0,
  "agent_reasoning": "High usage_gb (77.49849384877787 GB) exceeds the capacity of P_BASIC plan. Customer has dropped_calls (2) and complaints indicating dissatisfaction with service, suggesting a retention discount."
}


In [8]:
print("\nTesting CUST_013:")
print(run_agentic_workflow("CUST_013"))


Testing CUST_013:
{
  "recommended_plan_id": "P_STANDARD",
  "discount_applied_flag": false,
  "final_discount_amount": 0.0,
  "agent_reasoning": "High usage_gb (52.46002471889961) exceeds the capacity of P_BASIC plan, recommending upgrade to P STANDARD."
}


In [9]:
print("\nTesting CUST_376:")
print(run_agentic_workflow("CUST_376"))


Testing CUST_376:
{
  "recommended_plan_id": "P_ULTRA",
  "discount_applied_flag": true,
  "final_discount_amount": 10.0,
  "agent_reasoning": "High usage_gb (72.02471888726672 GB) exceeds the current plan's capacity, and customer has dropped_calls (1) and complaints indicating dissatisfaction."
}
